In [1]:
# =============================================================================
# Created By:     Kai Metzger
# Created School: Franz-Oberthuer-Schule Wuerzburg
# Created Email:  metzgerkai@franz-oberthuer-schule.de
# Created Date:   Mon Nov 01 21:02 UTC 2026
# Version:        1.0.2
# =============================================================================
"""The Module has been build for training the symbols dataset with the 
following datasets consisting of images + ground truth. Images have the base 
dimenstions of 640px x 480px in datasets 1-8 and then another 700 images with 
resolution 1920px x 1080px have been added to the datasets9+.
The files in the dataset <symbols> should be ordered in the following
manner starting with dataset10+:
- symbols      
  - # dataset e.g. datset10 the corresponding classes are stored in folders 0-3:
    #     classA/     = cross       361
    #     classB/     = circle      358
    #     classC/     = triangle    360
    #     classD/     = rectangle   361
"""

# =============================================================================
# Import
# =============================================================================
import cv2
import os
import numpy as np
from pathlib import Path
from keras import layers, models, utils, callbacks
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from keras.optimizers.schedules import ExponentialDecay
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import splitfolders

# Enable Nvidia GPUs by un-commenting this line 
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
XLA_FLAGS="--xla_gpu_cuda_data_dir=/usr/"

# To get reproducable results with the same training setting random seed
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

DEBUG = True

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# Declare variables / dataset
# =============================================================================
# Change the following paths to your dataset path
home_dir = os.path.expanduser("~")

dataset = ""                                            # CHANGE DATASET NUMBER

dataset_for_training = "dataset" + str(dataset)  
dataset_for_training_split = dataset_for_training + "-split70-15-15"
checkpoint_filepath = home_dir + "/ml_project/datasets/symbols/chpt/"
print(dataset_for_training)
print(checkpoint_filepath)

In [ ]:
# =============================================================================
# Split into 70% train, 15% val, 15% test on disk
# =============================================================================
source_dir = Path(dataset_for_training_split)
split_dir = source_dir.parent / f"{source_dir.name}"

if not split_dir.exists():
    splitfolders.ratio(
        input=str(dataset_for_training),
        output=str(split_dir),
        shuffle=True,
        seed=SEED,
        ratio=(0.70, 0.15, 0.15),
    )
else:
    print(f"Split folder already exists; skipping: {split_dir}")

In [ ]:
# =============================================================================
# Dataset related
# =============================================================================
NUM_CLASSES = 4                                     # change for other datasets
                                                    # e.g. 3 for rock, paper, 
                                                    # scissors
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
IMG_CHANNELS = 3                                       # 1=:grayscale, 3=:color
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

# =============================================================================
# Hyperparameters
# =============================================================================
BATCH_SIZE = 32
EPOCHS = 300
LEARNING_RATE = 1e-3

# Set up an exponential decay learning rate schedule
LR_SCHEDULER = ExponentialDecay(
    initial_learning_rate=LEARNING_RATE,    # start with 0.001
    decay_steps=500,                        # how often to apply the decay
    decay_rate=0.96,                        # decay rate
    staircase=True                          # whether to apply the decay in steps
)
OPTIMIZER_BASE = keras.optimizers.Adam(learning_rate=LR_SCHEDULER)

LR_SCHEDULER_FINE = ExponentialDecay(
    initial_learning_rate=LEARNING_RATE/10, # start with 0.001
    decay_steps=500,                        # how often to apply the decay
    decay_rate=0.96,                        # decay rate
    staircase=True                          # whether to apply the decay in steps
)
OPTIMIZER_FINE = keras.optimizers.RMSprop(learning_rate=LR_SCHEDULER_FINE)

In [ ]:
# =============================================================================
# Load dataset
# =============================================================================
split = "train-val-test"

if split == "train-val":
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        dataset_for_training,
        labels="inferred",
        label_mode="categorical",
        shuffle=True,
        validation_split=0.3,
        subset="training",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    validation_dataset = tf.keras.utils.image_dataset_from_directory(
        dataset_for_training,
        labels="inferred",
        label_mode="categorical",
        shuffle=False,
        validation_split=0.3,
        subset="validation",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
elif split == "train-val-test":
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        dataset_for_training_split + "/train",
        labels="inferred",
        label_mode="categorical",
        shuffle=True,
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    validation_dataset = tf.keras.utils.image_dataset_from_directory(
        dataset_for_training_split + "/val",
        labels="inferred",
        label_mode="categorical",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    test_dataset = tf.keras.utils.image_dataset_from_directory(
        dataset_for_training_split + "/test",
        labels="inferred",
        label_mode="categorical",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

In [ ]:
# =============================================================================
# Class names
# =============================================================================
class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print("Klassen:", class_names, "\n")

In [ ]:
# =============================================================================
# Preview training images
# =============================================================================
plt.figure(figsize=(8, 8))
for images, labels in train_dataset.take(1):
    for i in range(min(BATCH_SIZE, 9)):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_index = labels[i].numpy().argmax()
        plt.title(class_names[label_index])
        plt.axis("off")

plt.tight_layout()

In [ ]:
print('Number of training batches: %d' % tf.data.experimental.cardinality(train_dataset))
print('Number of validation batches: %d' % tf.data.experimental.cardinality(validation_dataset))

In [ ]:
# =============================================================================
# Prefetch datasets for performance
# =============================================================================
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

print('Number of testing batches: %d' % tf.data.experimental.cardinality(test_dataset))

In [ ]:
# =============================================================================
# Data Augmentation
# =============================================================================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.2),   
])

"""
# =============================================================================
# Full Data Augmentation Pipeline
# =============================================================================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),             # flip left/right
    layers.RandomRotation(0.1),                  # rotate ±10%
    layers.RandomZoom(0.1),                      # zoom in/out
    layers.RandomTranslation(0.1, 0.1),          # shift X/Y
    layers.RandomBrightness(factor=0.1),         # color jitter (TF 2.15+)
    layers.RandomContrast(0.1),                  # adjust contrast
    layers.RandomGaussianBlur(
        kernel_size=3,     # size of blur kernel (3, 5, etc.)
        sigma=(0.1, 2.0)   # range of possible blur intensity
    )  
])
"""

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(8, 8))

for first_image, _ in train_dataset.unbatch().take(1):
    for ax in axes.flat:
        augmented = data_augmentation(
            tf.expand_dims(first_image, axis=0),
            training=True,
        )
        ax.imshow((augmented[0] / 255).numpy())
        ax.axis("off")

plt.tight_layout()

In [ ]:
# AUTOTUNE
# =============================================================================
# TensorBoard Logging
# =============================================================================
log_dir = "logs/fit"
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

# Starten in Shell: tensorboard --logdir logs/fit

In [ ]:
# =============================================================================
# MobileNetV2 (Transfer Learning)
# =============================================================================
base_model = tf.keras.applications.MobileNetV2(
#base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

# Let's take a look at the base model architecture
#base_model.summary()

In [ ]:
# Build a model by chaining together the data augmentation, rescaling, base_model and feature extractor layers
inputs = keras.Input(shape=IMG_SIZE + (3,))

x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x) # best
#x = tf.keras.applications.mobilenet_v3.preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR_SCHEDULER),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# =============================================================================
# Early stopping after e.g. 10 epochs if loss does not improve
# =============================================================================
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
)

In [ ]:
# =============================================================================
# Callback for training
# =============================================================================
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = Path(checkpoint_filepath)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_file = checkpoint_dir / f"chpt_{timestamp}.keras"

print(checkpoint_file)  # Should end with chpt_YYYYMMDD_HHMMSS.keras

# Set verbose flag to 0 to omit loss/accuracy output per epoch
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath=str(checkpoint_file),
    monitor='val_loss',       # metric to monitor
    save_best_only=True,      # only save when metric improves
    mode='auto',              # minimize loss and aximize accuracy
    verbose=0,                # Print a line when the model is saved
)

In [ ]:
# =============================================================================
# Training (Feature Extraction)
# =============================================================================
initial_epochs = 100 # best: 50

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=initial_epochs,
    shuffle=True,
    callbacks=[checkpoint,
               early_stopping,
               tensorboard_callback],
)

# Starten in Shell: tensorboard --logdir logs/fit

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0,1.0])
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
# =============================================================================
# Fine Tuning
# =============================================================================
# Let's take a look to see how many layers are in the base model
print("Number of layers in the base model: ", len(base_model.layers))

base_model.trainable = True

# Fine-tune from this layer onwards
fine_tune_from = 100

for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.RMSprop(LR_SCHEDULER_FINE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_epochs = 100 # best: 100
total_epochs = initial_epochs + fine_tune_epochs

history_fine = model.fit(train_dataset,
                         validation_data=validation_dataset,
                         epochs=total_epochs,
                         initial_epoch=len(history.epoch),
                         shuffle=True,
                         callbacks=[checkpoint,
                                    tensorboard_callback],
)

In [ ]:
acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']

loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

In [ ]:
plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.ylim([0.6, 1])
plt.plot([initial_epochs-1,initial_epochs-1],
          plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.ylim([0, 1.0])
plt.plot([initial_epochs-1,initial_epochs-1],
         plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
# =============================================================================
# Test Accuracy on test set
# =============================================================================
results = model.evaluate(test_dataset, 
                         verbose=1, 
                         return_dict=True,
                         )

loss = results["loss"]
accuracy = results.get("accuracy")  # available if you compiled with an accuracy metric

print("Test loss:", loss)
print("Test accuracy:", accuracy)

In [ ]:
# =============================================================================
# Konfusionsmatrix + Klassifizierungsreport
# =============================================================================
# Daten aus Validation/Test holen
y_true = []
y_pred = []

for images, labels in validation_dataset:
    preds = model.predict(images)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Klassenliste
print("Klassen:", class_names)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
loss, accuracy = model.evaluate(test_dataset)
print('Test accuracy :', accuracy)

In [ ]:
plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Test Accuracy')
plt.ylim([0.8, 1])
plt.plot([initial_epochs-1,initial_epochs-1],
          plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='lower right')
plt.title('Training and Test Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Test Loss')
plt.ylim([0, 1.0])
plt.plot([initial_epochs-1,initial_epochs-1],
         plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='upper right')
plt.title('Training and Test Loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
# =============================================================================
# Grad-CAM Visualisierung
# =============================================================================

# ============================================================
# 1. Extract MobileNetV2 backbone
# ============================================================
backbone = base_model

# Find last conv layer inside MobileNetV2
last_conv = None
for layer in reversed(backbone.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv = layer
        break

print("Last conv layer:", last_conv.name)


# ============================================================
# 2. Create a Grad-CAM model for MobileNetV2 only
# ============================================================
grad_model = tf.keras.models.Model(
    inputs=backbone.input,            # internal input of MobileNetV2
    outputs=[last_conv.output,        # feature map
             backbone.output]         # MobileNetV2 embeddings
)


# ============================================================
# 3. Extract and preprocess image manually (bypass Sequential)
# ============================================================
example_img, _ = next(iter(validation_dataset))
example_img = example_img[0].numpy() # shape: (H,W,3)

# Your model expects raw 0–255
if example_img.max() <= 1.0:
    example_img = example_img * 255.

# Apply SAME preprocessing as your Sequential layer
# Apply Rescaling(1./127.5, offset=-1) manually
img_pp = (example_img / 127.5) - 1

# final MobileNetV2 input
input_backbone = np.expand_dims(img_pp, 0)


# ============================================================
# 4. Run Grad-CAM on the backbone
# ============================================================
with tf.GradientTape() as tape:
    conv_outputs, features = grad_model(input_backbone)
    conv_outputs = conv_outputs[0]

# use full model for final class prediction
full_pred = model(np.expand_dims(example_img, 0))
pred_index = tf.argmax(full_pred[0]).numpy()
print(full_pred)

# MobileNetV2 embeddings → Dense layer → final class  
dense_weights = model.layers[-1].get_weights()[0]     # (1280, num_classes)
weights_for_class = dense_weights[:, pred_index]       # shape (1280,)

# Compute Grad-CAM manually from MobileNetV2 embeddings
grads = weights_for_class
heatmap = tf.reduce_sum(tf.multiply(conv_outputs, grads), axis=-1)

heatmap = np.maximum(heatmap, 0)
heatmap /= heatmap.max() + 1e-8


# ============================================================
# 5. Visualization
# ============================================================
img_uint8 = example_img.astype("uint8")

heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
heatmap_uint8 = np.uint8(255 * heatmap_resized)
heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

overlay = cv2.addWeighted(img_uint8, 0.6, heatmap_color, 0.4, 0)


plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.title("Original")
plt.imshow(img_uint8)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Grad-CAM Heatmap")
plt.imshow(heatmap_resized, cmap='jet')
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Overlay")
plt.imshow(overlay)
plt.axis("off")

plt.show()

print("Predicted class:", class_names[pred_index])


In [ ]:
# =============================================================================
# Modell speichern
# =============================================================================
keras.saving.save_model(model, "model.keras")

print("Modelle gespeichert: model.keras")

# =============================================================================
# Modell laden
# =============================================================================
loaded_model = keras.models.load_model("model.keras")

print("Modelle erfolgreich geladen!")